### Contents:
ChromaDB w HuggingFace <br>
ChromaDB w LCEL (LangChain Expression Language)

### RAG system with LangChain and ChromaDB

RAG is a technique that combines capabilities of LLMs w external knowledge retrieval.

ChromaDB: An open source vector database for storing and retrieving embeddings. It is a vector store.

RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
#Langchain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.schema import Document

#Vector Stores imports
from langchain_community.vectorstores import Chroma

#utility imports
import numpy as np
from typing import List


d:\project\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Sample Data

In [3]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [4]:
# save sample doc to files

import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)


In [5]:
# # save sample doc to files

# import tempfile
# temp_dir=tempfile.mkdtemp()

# for i,doc in enumerate(sample_docs):
#     with open(f"{temp_dir}/doc_{i}.txt","w") as f:
#         f.write(doc)

# print(f"Sample doc created in : {temp_dir}")

# import shutil

# delete the entire temp directory and its contents
# shutil.rmtree(temp_dir)

# print(f"Deleted directory: {temp_dir}")


### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

#load all the text files from the directory
dir_loader=DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents=dir_loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst Document preview:")
print(documents[0].page_content[:200]+'...')

Loaded 3 documents

First Document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


### 3. Document Splitting

In [7]:
#Initialize TextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len, #how to measure chunks size
    separators=["\n\n","\n",". "," ",""] #Hirearchy of separators
)
chunks=text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} docs")
print(f"First Chunk: {chunks[4].page_content[:400]}...")
print(f"Metadata {chunks[0].metadata}")

Created 7 chunks from 3 docs
First Chunk: Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, wh...
Metadata {'source': 'data\\doc_0.txt'}


In [8]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep l

### 4. Embedding Models

In [ ]:
sample_text="MAchine LEARNing is fascinating"
embeddings=HuggingFaceEmbeddings(
    model_name="MODEL_NAME"
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [10]:
embeddingss=embeddings.embed_query(sample_text)
embeddingss


[-0.016979273408651352,
 0.0791153535246849,
 -0.05364146828651428,
 -0.004374045412987471,
 -0.03569667413830757,
 0.033180687576532364,
 -0.00898682326078415,
 -0.01875317469239235,
 -0.03127491846680641,
 0.013532880693674088,
 0.025487802922725677,
 0.06858398020267487,
 -0.0336262471973896,
 0.05780269205570221,
 0.00992253702133894,
 -0.07284895330667496,
 0.0012737527722492814,
 -0.01181392278522253,
 -0.05102642625570297,
 0.0029895754996687174,
 -0.04443347081542015,
 -0.035726092755794525,
 -0.002910713432356715,
 -0.0006069194641895592,
 0.012111497111618519,
 -0.025453047826886177,
 0.008542343974113464,
 -0.018963919952511787,
 -0.006859422195702791,
 -0.007028596941381693,
 -0.024332696571946144,
 -0.028051767498254776,
 -0.022490790113806725,
 0.08795143663883209,
 1.4996801382949343e-06,
 -0.050870381295681,
 -0.004036599304527044,
 0.016153352335095406,
 -0.058273158967494965,
 0.026826031506061554,
 0.06077723950147629,
 0.029385387897491455,
 -0.001340847578831017,
 

### 5. Initialize ChromaDB Vector Store 
Store chunks in vector representation

In [11]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

#create a chromadb vector store
persist_directory="./chroma_db"

#initialize chromadb w huggingface
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory, #directory to store the data
    collection_name="rag_collections"

)

print(f"Vector Store created w {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

Vector Store created w 49 vectors
Persisted to : ./chroma_db


#### 6. Text Similarity Search

In [12]:
query="what is machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinfor

In [13]:
queryy="what is NLP?"

similar_docs=vectorstore.similarity_search(queryy,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mec

In [14]:
print(f"Query:{query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i,doc in enumerate(similar_docs):
    print(f"\n---Chunk{i+1}---")
    print(doc.page_content[:200]+'...')
    print(f"source:{doc.metadata.get('source','Unknown')}")

Query:what is machine learning?

Top 3 similar chunks:

---Chunk1---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
source:data\doc_2.txt

---Chunk2---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
source:data\doc_2.txt

---Chunk3---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
source:data\doc_2.txt


### 7. Advanced Similarity Seach with Scores

In [15]:
results_score=vectorstore.similarity_search_with_score(query,k=3)
results_score

[(Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  0.46769100427627563),
 (Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns 

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### 8. Initialize LLM, RAG Chain, Prompt Template, Query the RAG System

In [16]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation", #model to generate text based on an input prompt
    model="HuggingFaceH4/zephyr-7b-beta",
    max_new_tokens=256, #the output will stop after generating up to 512 tokens.
    temperature=0.7,
    device_map="auto", #Assigns the model to GPU(s) or CPU automatically. 
                       #"auto" lets Hugging Face detect available hardware and split the model if needed.
    trust_remote_code=True
)

llm = HuggingFacePipeline(pipeline=pipe)


Loading checkpoint shards: 100%|██████████| 8/8 [00:00<00:00, 383.94it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cpu
C:\Users\vuaku\AppData\Local\Temp\ipykernel_13552\3213159523.py:14: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


temperature=0.7

Controls randomness/creativity in generation.

Range: 0 (deterministic) → 1.0 (more random).

Example:

0 → model always picks the most likely next token.

0.7 → some randomness, more natural or creative responses.

Typical values: 0.0–0.8 for RAG / QA tasks.

In [17]:
# text_response=llm.invoke("What is Large Language Models?")
# text_response

### Modern RAG Chain

In [18]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

In [19]:
#convert vector store to retriever

retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} #Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000190F087F8C0>, search_kwargs={})

In [20]:
# Create a prompt template
from langchain_core.prompts import ChatPromptTemplate

system_prompt="""You are a assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt=ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

The output is following the above chain template. Following the system prompt, and then the human input.

In [21]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [22]:
#Create a document chain
# a document chain is used to combine all the retrived relevant information
# and stuff it inside the LLM

from langchain.chains.combine_documents import create_stuff_documents_chain
doc_chain=create_stuff_documents_chain(llm,prompt)
doc_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| HuggingFacePipeline(pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x0000019084648050>)
| StrOutputPars

How the chain works:
- Takes retrived docs
- Stuffs them into the prompt's {context} placeholder
- Sends the complete prompt prompt to the LLM
- Returns the LLM's response

In [23]:
# Creates the final RAG chain

from langchain.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(retriever,doc_chain) 

rag_chain

#retriever fetch info, doc_chain process the relevant info
# replace the doc_chain into the {context} placeholder of prompt

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000190F087F8C0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you 

We need two chains. <br>
One chain is used to combine all the relevant information - create_stuff_document_chain<br>
Other chain is used to retrieve the information, it combines the retriever and the first doc chain- create_retrieval_chain<br>
Then, both chains are combined to the LLM to provide an output. Send a call

In [24]:
#Call the LLM
# response=rag_chain.invoke({"input":"What is Deep learning"})

In [25]:
# response

NameError: name 'response' is not defined

In [ ]:
# response['answer']

"System: You are a assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers\n\nDeep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    proces

🔸 Summary Flow

sentence-transformers/all-mpnet-base-v2
→ Converts docs & query into embeddings → stored in ChromaDB

ChromaDB
→ Finds the most similar chunks for the query.

HuggingFaceH4/zephyr-7b-beta
→ Reads the retrieved chunks + query → generates the final answer.



🧩 Why not use the same model for both?

Because:

Embedding models are optimized for semantic similarity, not generation.

LLMs are optimized for text generation, not vector encoding.

RAG combines both strengths:

Embedding model for retrieval,
LLM for generation → Retrieval-Augmented Generation ✅

In [ ]:
# #Function to query the modern RAG system

# def query_rag_modern(question):
#     print(f"Questiion:{question}")
#     print("-"*50)

#     #Using create_retrieval_chain approach
#     result=rag_chain.invoke({"input":question})
#     print(f"Answer:{result['answer']}")
#     print("\nRetrieved Context:")
#     for i,doc in enumerate(result['context']):
#         print(f"\n---Source{i+1}---")
#         print(doc.page_content[:200]+'...')

#     return result 

# #Test Queries
# test_q=[
#     "What are the three types of machine learning?",
#     "What is deep learning and how does it relate to neural networls?",
#     "What are CNNs best used for?"
# ]

# for question in test_q:
#     result=query_rag_modern(question)
#     print("\n"+"="*80+"\n")


Questiion:What are the three types of machine learning?
--------------------------------------------------


#### Building RAG using LCEL (LangChain Expression Language)

In [26]:

#Very Flexible approach using LCEL

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel



StrOutputParser - tells the LLM how the output should be generated <br>
RunnablePassthrough - In a specific chain, it is responsible for the info from one stage to another<br>
RunnableParallel - Responsible for running multiple chains together 

In [27]:
#create a custom prompt
from langchain_core.prompts import ChatPromptTemplate



custom_prompt=ChatPromptTemplate.from_template("""
Use the following context to answer the question. If you don;t know the answer based on the context, say you don't know. Provide specific details from the context to support your answer.
Context : {context}
Question : {question}
Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nUse the following context to answer the question. If you don;t know the answer based on the context, say you don't know. Provide specific details from the context to support your answer.\nContext : {context}\nQuestion : {question}\nAnswer:"), additional_kwargs={})])

In [28]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000190F087F8C0>, search_kwargs={})

In [29]:
#Format the output doc for the prompt

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [30]:
#Build the chain using LCEL

rag_chain_lcel=(
    {"context":retriever| format_docs,
     "question":RunnablePassthrough(),
     
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000190F087F8C0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nUse the following context to answer the question. If you don;t know the answer based on the context, say you don't know. Provide specific details from the context to support your answer.\nContext : {context}\nQuestion : {question}\nAnswer:"), additional_kwargs={})])
| HuggingFacePipeline(pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x0000019084648050>)
| StrOutputParser()

In [ ]:
response=rag_chain_lcel.invoke("What is Deep Learning?")
response

Whenever we use RunnablePassthrough(), we hav to give the string directly here<br>
response=rag_chain_lcel.invoke("What is Deep Learning?") <br><br>
No need of "input":{context} tht bs

In [ ]:
retriever.get_relevant_documets("What is deep learning?")

In [42]:
#Query using the LCEL approach - Fixed Version
def query_rag_lcel(question):
    print(f"Question:{question}")
    print("-"*50)

    #Method 1: Pass string directly (when using RunnablePassthrough())
    answer=rag_chain_lcel.invoke(question)
    print(f"Answer:{answer}")

    #Get source documents seperatle if needed
    docs=retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i,doc in enumerate(docs):
        print(f"\n---Source {i+1} ----")
        print(doc.page_content[:200]+"...")

In [ ]:
query_rag_lcel("What are the key concepts in reinforced learning?")

### Add new documents to existing vector store

In [33]:
new_doc="""
Reinforcement Learning in detail
Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environent. The agent receives rewards or penalties based 
on its actions and learns to maximize cumulative reward over time. Key concepts in RL include:
states, actions, rewards, policies, and value functions. Popular RL algorithms include Q-learning, Deep-Q-Networks(DQN), Policy Gradient Methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [34]:
new_doc

'\nReinforcement Learning in detail\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environent. The agent receives rewards or penalties based \non its actions and learns to maximize cumulative reward over time. Key concepts in RL include:\nstates, actions, rewards, policies, and value functions. Popular RL algorithms include Q-learning, Deep-Q-Networks(DQN), Policy Gradient Methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n'

In [35]:
new_document=Document(
    page_content=new_doc,
    metadata={"source":"manual_addition","topic":"reinforcement_learing"}
)

In [36]:
new_document

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learing'}, page_content='\nReinforcement Learning in detail\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environent. The agent receives rewards or penalties based \non its actions and learns to maximize cumulative reward over time. Key concepts in RL include:\nstates, actions, rewards, policies, and value functions. Popular RL algorithms include Q-learning, Deep-Q-Networks(DQN), Policy Gradient Methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n')

In [38]:
#text splitter

new_chunks=text_splitter.split_documents([new_document])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learing'}, page_content='Reinforcement Learning in detail\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environent. The agent receives rewards or penalties based \non its actions and learns to maximize cumulative reward over time. Key concepts in RL include:'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learing'}, page_content='on its actions and learns to maximize cumulative reward over time. Key concepts in RL include:\nstates, actions, rewards, policies, and value functions. Popular RL algorithms include Q-learning, Deep-Q-Networks(DQN), Policy Gradient Methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.')]

In [39]:
#Add new documents to vector store
vectorstore.add_documents(new_chunks)

['ca2b3736-906c-4b00-a257-8507a677b144',
 '07eed4fc-e3c1-4aa7-aea4-2b1abebbfdd3']

In [40]:
print(f"Added {len(new_chunks)} new chunks to the vector store.")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store.
Total vectors now: 51


In [ ]:
#query with updated vector
new_question="What are the key concepts in reinforcement learning?"
result=query_rag_lcel(new_question)

Question:What are the key concepts in reinforcement learning?
--------------------------------------------------


#### Advanced RAG Techniques - Conversational Memory

In [46]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

- create_history_aware_retriever: Make the retriever understand conversation context
- MessagesPlaceholder : Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [47]:
#create prompt that includes chat history

contextualize_q_system_prompt=""" 
Given a chat history and the latest user question which might
reference context in the chat history, formulate a standalone 
question which can be understood without the chat history. Do 
NOT answer the question, just remember it if needed and otherwise 
return it as it is.
"""
contextualize_q_prompt=ChatPromptTemplate.from_messages([
    ("system",contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])


In [48]:
#Create history aware retriever
history_aware_retriever=create_history_aware_retriever(
    llm,retriever,contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000190F087F8C0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag

In [54]:
#Create a new document chain with history
qa_system_prompt=""" 
You are an assistant for question-answering tasks. Use the following pieces of retrieved
context to answer the question. If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
"""

qa_prompt=ChatPromptTemplate([
    ("system",qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","Context:\n{context}\n\nQuestion:\n{input}")
])

qa_answer_chain=create_stuff_documents_chain(llm,qa_prompt)

conversational_rag_chain=create_retrieval_chain(
    history_aware_retriever,
    qa_answer_chain
)

print("Conversational RAG chain created.")

Conversational RAG chain created.


In [ ]:
chat_history=[]

#First Question
result1=conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"What is machine learning?"
})

print(f"Q:What is machine learning?")
print(f"A:{result1['answer']}")

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [ ]:
chat_history

In [ ]:
#Follow up question

result2=conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"What are its types?"  #Refers to the prev question
})
print("Q:What are its types?")
print(f"A:{result2['answer']}")